# Giant Comparison Table

Six rows × four scenarios × five metrics, aggregated over 5 seeds.

**Methods (rows):**
1. **SF** — Social Force baseline (`trained_models/SF`)
2. **ORCA** — ORCA baseline (`trained_models/ORCA`)
3. **CrowdNav++** — GST predictor baseline (`trained_models/GST_predictor_rand`)
4. **GenSafeNav (conservative)** — LoraF rank 1 with LoRA branch **off** (`always_off`)
5. **GenSafeNav (aggressive)** — Full fine-tune on the invi→visi setup (`FullFineTune_invi_visi`, `always_off`)
6. **Ours (adaptive GT)** — LoraF rank 1 with `adaptive_gt` behaviour

**Scenarios (columns):** seperate_all_aware, seperate_all_ignorant, seperate_mixed_5050, cluster_aware_ignorant.

**Metrics:** Success Rate (SR), Collision Rate (CR), Nav Time (NT), Path Length (PL), Intrusion Time Ratio (ITR). SR/CR/ITR shown as percent. Mean and seed-SD are reported in separate columns across 5 seeds (42, 1000, 2000, 3000, 4000). Wide tables are grouped first by scenario, then metric, then statistic.

In [1]:
import os, json
from collections import defaultdict
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ----------------------- config -----------------------
SCENARIOS = [
    'seperate_all_ignorant',
    'seperate_all_aware',
    'seperate_mixed_5050',
    'cluster_aware_ignorant',
]

SCENARIO_LABELS = {
    'seperate_all_ignorant': 'Non-aware',
    'seperate_all_aware': 'Aware',
    'seperate_mixed_5050': 'Mixed',
    'cluster_aware_ignorant': 'Spatial Clusters',
}
SCENARIO_DISPLAY_ORDER = SCENARIOS

# Baseline seeds use the EXP_ID_seed=N_N pattern from the patched
# test_baselines.sh. LoRA-based rows use plain seed labels.
SEEDS_BASELINE = {'1000_1000', '2000_2000', '3000_3000', '4000_4000'}
SEEDS_LORA     = {'42', '1000', '2000', '3000', '4000'}

# (display_name, source_dir, behaviour_suffix, seed_set)
METHODS = [
    ('SF',                       'trained_models/SF',                       'always_off',  SEEDS_BASELINE),
    ('ORCA',                     'trained_models/ORCA',                     'always_off',  SEEDS_BASELINE),
    ('CrowdNav++',               'trained_models/GST_predictor_rand',       'always_off',  SEEDS_BASELINE),
    ('GenSafeNav (conservative)','trained_models/LoraF_invi_visi_rank_1',   'always_off',  SEEDS_LORA),
    ('GenSafeNav (aggressive)',  'trained_models/FullFineTune_invi_visi',   'always_off',  SEEDS_BASELINE),
    ('Ours (adaptive GT)',       'trained_models/LoraF_invi_visi_rank_1',   'adaptive_gt', SEEDS_LORA),
]

# metric key in summary → (display name, in_percent, decimals)
METRICS = [
    ('success_rate',           'SR (%)',   True,  2),
    ('collision_rate',         'CR (%)',   True,  2),
    ('avg_nav_time',           'NT (s)',   False, 2),
    ('avg_path_length',        'PL (m)',   False, 2),
    ('avg_intrusion_ratio_pct','ITR (%)',  False, 2),  # already in percent
    ('avg_min_social_distance','SocD (m)', False, 3),  # mean per-episode min human–robot distance
    ('avg_inference_time_ms', 'Inf (ms)', False, 3),
    ('max_inference_peak_gpu_memory_mb', 'Peak GPU (MB)', False, 1),
    ('avg_matrix_calc_time_ms', 'Mat (ms)', False, 3),
    ('num_episodes', 'N', False, 0),
]

In [2]:
# ----------------------- load -----------------------
def load_method(model_dir, behaviour, seed_set):
    """Return {scenario: {seed: {metric: value}}}."""
    path = os.path.join(model_dir, 'test', 'all_evaluations.json')
    out = {sc: {} for sc in SCENARIOS}
    if not os.path.exists(path):
        return out
    d = json.load(open(path))
    for k, v in d.items():
        if '_exp' not in k:
            continue
        base, exp = k.rsplit('_exp', 1)
        if exp not in seed_set:
            continue
        if not base.endswith('_' + behaviour):
            continue
        sc = base[:-(len(behaviour)+1)]
        if sc not in SCENARIOS:
            continue
        out[sc][exp] = v.get('summary', {})
    return out

loaded = {name: load_method(d, beh, seeds) for name, d, beh, seeds in METHODS}

# Coverage report
print(f"{'method':<28} {'scenario':<26} {'expected':<5}  seeds_found")
for name, _, _, seeds in METHODS:
    for sc in SCENARIOS:
        found = sorted(loaded[name][sc].keys())
        marker = '✓' if seeds.issubset(found) else 'PARTIAL' if found else 'EMPTY'
        print(f"  {name:<26} {sc:<26} {len(seeds):<5}  {marker:<8} {found}")

method                       scenario                   expected  seeds_found
  SF                         seperate_all_ignorant      4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  SF                         seperate_all_aware         4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  SF                         seperate_mixed_5050        4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  SF                         cluster_aware_ignorant     4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       seperate_all_ignorant      4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       seperate_all_aware         4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       seperate_mixed_5050        4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       cluster_aware_igno

In [3]:
# ----------------------- aggregate → separate mean / seed-SD columns -----------------------
def fmt_stat(value, decimals):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return '—'
    return f"{value:.{decimals}f}"


def metric_stat_columns(metrics):
    cols = []
    for _, label, _, _ in metrics:
        if label == 'N':
            cols.append((label, 'total'))
        else:
            cols.extend([(label, 'mean'), (label, 'std')])
    return cols


def add_metric_stats(row, metric_key, label, in_pct, decimals, vals):
    if not vals:
        if label == 'N':
            row[f'{label} total'] = '—'
        else:
            row[f'{label} mean'] = '—'
            row[f'{label} std'] = '—'
        return

    if metric_key == 'num_episodes':
        row[f'{label} total'] = str(int(np.sum(vals)))
        return

    mean = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    if in_pct:
        mean *= 100
        sd *= 100
    row[f'{label} mean'] = fmt_stat(mean, decimals)
    row[f'{label} std'] = fmt_stat(sd, decimals)


rows = []
for name, _, _, _ in METHODS:
    for sc in SCENARIOS:
        seed_dicts = loaded[name][sc]
        row = {'Method': name, 'Scenario': sc}
        for mkey, mlabel, in_pct, dec in METRICS:
            vals = [s.get(mkey) for s in seed_dicts.values() if s.get(mkey) is not None]
            add_metric_stats(row, mkey, mlabel, in_pct, dec, vals)
        rows.append(row)

df = pd.DataFrame(rows)
df


,Method,Scenario,SR (%) mean,SR (%) std,CR (%) mean,CR (%) std,NT (s) mean,NT (s) std,PL (m) mean,PL (m) std,ITR (%) mean,ITR (%) std,SocD (m) mean,SocD (m) std,Inf (ms) mean,Inf (ms) std,Peak GPU (MB) mean,Peak GPU (MB) std,Mat (ms) mean,Mat (ms) std,N total
0,SF,seperate_all_ignorant,13.60,2.47,20.40,3.88,30.05,1.01,35.15,0.67,3.44,0.38,0.429,0.007,—,—,—,—,—,—,1000
1,SF,seperate_all_aware,18.80,2.40,0.20,0.23,28.80,0.97,40.90,0.76,1.52,0.14,0.437,0.022,—,—,—,—,—,—,1000
2,SF,seperate_mixed_5050,15.60,2.99,11.20,2.99,28.39,1.64,38.06,1.05,2.65,0.58,0.429,0.012,—,—,—,—,—,—,1000
3,SF,cluster_aware_ignorant,13.80,1.74,0.60,0.52,28.53,3.49,39.74,0.31,0.34,0.09,0.418,0.029,—,—,—,—,—,—,1000
4,ORCA,seperate_all_ignorant,66.40,3.88,29.20,2.71,23.68,0.82,20.01,0.40,1.04,0.16,0.492,0.009,—,—,—,—,—,—,1000
5,ORCA,seperate_all_aware,93.10,1.10,0.70,0.20,23.32,0.72,22.98,0.49,0.54,0.08,0.478,0.017,—,—,—,—,—,—,1000
6,ORCA,seperate_mixed_5050,83.70,1.91,11.80,1.48,23.87,0.41,21.98,0.32,0.98,0.14,0.488,0.013,—,—,—,—,—,—,1000
7,ORCA,cluster_aware_ignorant,89.60,2.24,6.90,1.89,18.94,0.51,21.29,0.48,0.59,0.17,0.472,0.023,—,—,—,—,—,—,1000
8,CrowdNav++,seperate_all_ignorant,89.20,2.36,10.80,2.36,13.73,0.26,19.99,0.26,7.96,0.46,0.416,0.006,—,—,—,—,—,—,1000
9,CrowdNav++,seperate_all_aware,99.60,0.46,0.40,0.46,11.97,0.17,19.00,0.10,7.29,0.50,0.417,0.011,—,—,—,—,—,—,1000


In [4]:
# ----------------------- compact pivoted view: rows=Method, columns=(Scenario, Metric, Stat) -----------------------
value_cols = [c for c in df.columns if c not in ('Method', 'Scenario')]
long = df.melt(id_vars=['Method', 'Scenario'], value_vars=value_cols, var_name='MetricStat', value_name='Value')
long[['Metric', 'Stat']] = long['MetricStat'].str.rsplit(' ', n=1, expand=True)
long['Scenario group'] = long['Scenario'].map(SCENARIO_LABELS).fillna(long['Scenario'])
wide = long.pivot_table(index='Method', columns=['Scenario group', 'Metric', 'Stat'], values='Value', aggfunc='first')
wide = wide.reindex([m[0] for m in METHODS])
wide = wide.reindex(columns=pd.MultiIndex.from_tuples(
    [(SCENARIO_LABELS[sc], metric, stat) for sc in SCENARIO_DISPLAY_ORDER for metric, stat in metric_stat_columns(METRICS)],
    names=['Scenario', 'Metric', 'Stat'],
))
wide


Scenario                  Non-aware                                                                                                                                  Aware                            \
Metric                       SR (%)       CR (%)       NT (s)       PL (m)       ITR (%)       SocD (m)        Inf (ms)        Peak GPU (MB)     Mat (ms)         N SR (%)       CR (%)       NT (s)   
Stat                           mean   std   mean   std   mean   std   mean   std    mean   std     mean    std     mean    std          mean std     mean std total   mean   std   mean   std   mean   
Method                                                                                                                                                                                                 
SF                            13.60  2.47  20.40  3.88  30.05  1.01  35.15  0.67    3.44  0.38    0.429  0.007        —      —             —   —        —   —  1000  18.80  2.40   0.20  0.23  28.80   
ORCA                          66.40  3.88  29.20  2.71  23.68  0.82  20.01  0.40    1.04  0.16    0.492  0.009        —      —             —   —        —   —  1000  93.10  1.10   0.70  0.20  23.32   
CrowdNav++                    89.20  2.36  10.80  2.36  13.73  0.26  19.99  0.26    7.96  0.46    0.416  0.006        —      —             —   —        —   —  1000  99.60  0.46   0.40  0.46  11.97   
GenSafeNav (conservative)         —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —      —     —      —     —      —   
GenSafeNav (aggressive)           —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —      —     —      —     —      —   
Ours (adaptive GT)            97.04  0.36   2.96  0.36  16.51  0.44  23.44  0.59    2.39  0.25    0.448  0.005    5.856  0.199             —   —        —   —  1250  99.28  0.33   0.72  0.33  10.94   

Scenario                                                                                                                         Mixed                                                             \
Metric                          PL (m)       ITR (%)       SocD (m)        Inf (ms)        Peak GPU (MB)     Mat (ms)         N SR (%)       CR (%)       NT (s)       PL (m)       ITR (%)         
Stat                        std   mean   std    mean   std     mean    std     mean    std          mean std     mean std total   mean   std   mean   std   mean   std   mean   std    mean   std   
Method                                                                                                                                                                                              
SF                         0.97  40.90  0.76    1.52  0.14    0.437  0.022        —      —             —   —        —   —  1000  15.60  2.99  11.20  2.99  28.39  1.64  38.06  1.05    2.65  0.58   
ORCA                       0.72  22.98  0.49    0.54  0.08    0.478  0.017        —      —             —   —        —   —  1000  83.70  1.91  11.80  1.48  23.87  0.41  21.98  0.32    0.98  0.14   
CrowdNav++                 0.17  19.00  0.10    7.29  0.50    0.417  0.011        —      —             —   —        —   —  1000  95.40  1.24   4.60  1.24  13.06  0.20  19.75  0.23    7.36  0.73   
GenSafeNav (conservative)     —      —     —       —     —        —      —        —      —             —   —        —   —     —      —     —      —     —      —     —      —     —       —     —   
GenSafeNav (aggressive)       —      —     —       —     —        —      —        —      —             —   —        —   —     —      —     —      —     —      —     —      —     —       —     —   
Ours (adaptive GT)         0.11  18.16  0.28    7.33  0.74    0.417  0.011    3.767  0.219             —   —        —   —  1250  95.44  1.28   4.56  1.28  13.68  0.39  20.66  0.54    5.87  0.38   

Scenario                    

In [5]:
# ============================================================
# Table A — SD ACROSS SCENARIOS (per method × metric)
#   cell = std over the 4 scenario means (each scenario mean is itself
#   the mean over its seed-set). Big number → method is sensitive to scenario.
# ============================================================
def per_method_scenario_seed_values(name, scenario, mkey):
    return [s.get(mkey) for s in loaded[name][scenario].values() if s.get(mkey) is not None]

rows_a = []
for name, _, _, _ in METHODS:
    row = {'Method': name}
    for mkey, mlabel, in_pct, dec in METRICS:
        scen_means = []
        for sc in SCENARIOS:
            vals = per_method_scenario_seed_values(name, sc, mkey)
            if vals:
                scen_means.append(float(np.mean(vals)))
        if len(scen_means) > 1:
            sd = float(np.std(scen_means, ddof=1))
            if in_pct: sd *= 100
            row[mlabel] = round(sd, dec)
        else:
            row[mlabel] = np.nan
    rows_a.append(row)

df_sd_scenarios = pd.DataFrame(rows_a).set_index('Method')
df_sd_scenarios

,SR (%),CR (%),NT (s),PL (m),ITR (%),SocD (m),Inf (ms),Peak GPU (MB),Mat (ms),N
Method,,,,,,,,,,
SF,2.41,9.65,0.76,2.50,1.35,0.008,NaN,NaN,NaN,0.0
ORCA,11.85,12.24,2.35,1.25,0.26,0.009,NaN,NaN,NaN,0.0
CrowdNav++,4.29,4.29,0.74,0.43,0.96,0.004,NaN,NaN,NaN,0.0
GenSafeNav (conservative),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GenSafeNav (aggressive),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ours (adaptive GT),1.95,1.95,2.32,2.20,2.17,0.013,1.881,NaN,NaN,0.0


In [6]:
# ============================================================
# Table B — SD ACROSS EVAL SEEDS (per method × scenario × metric)
#   cell = std over the seeds, kept SEPARATE per scenario (not averaged).
#   Rows = Method; Columns = MultiIndex(Scenario, Metric).
# ============================================================
rows_b = []
for name, _, _, _ in METHODS:
    row = {'Method': name}
    for sc in SCENARIOS:
        for mkey, mlabel, in_pct, dec in METRICS:
            vals = per_method_scenario_seed_values(name, sc, mkey)
            if len(vals) > 1:
                sd = float(np.std(vals, ddof=1))
                if in_pct: sd *= 100
                row[(sc, mlabel)] = round(sd, dec)
            else:
                row[(sc, mlabel)] = np.nan
    rows_b.append(row)

df_sd_seeds = pd.DataFrame(rows_b).set_index('Method')
df_sd_seeds.columns = pd.MultiIndex.from_tuples(df_sd_seeds.columns, names=['Scenario', 'Metric'])
df_sd_seeds = df_sd_seeds.reindex(columns=pd.MultiIndex.from_product(
    [SCENARIOS, [m[1] for m in METRICS]], names=['Scenario', 'Metric']))
df_sd_seeds

Scenario                  seperate_all_ignorant                                                                            seperate_all_aware                                                 \
Metric                                   SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m) Inf (ms) Peak GPU (MB) Mat (ms)    N             SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m) Inf (ms)   
Method                                                                                                                                                                                         
SF                                         2.47   3.88   1.01   0.67    0.38    0.007      NaN           NaN      NaN  0.0               2.40   0.23   0.97   0.76    0.14    0.022      NaN   
ORCA                                       3.88   2.71   0.82   0.40    0.16    0.009      NaN           NaN      NaN  0.0               1.10   0.20   0.72   0.49    0.08    0.017      NaN   
CrowdNav++                                 2.36   2.36   0.26   0.26    0.46    0.006      NaN           NaN      NaN  0.0               0.46   0.46   0.17   0.10    0.50    0.011      NaN   
GenSafeNav (conservative)                   NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                NaN    NaN    NaN    NaN     NaN      NaN      NaN   
GenSafeNav (aggressive)                     NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                NaN    NaN    NaN    NaN     NaN      NaN      NaN   
Ours (adaptive GT)                         0.36   0.36   0.44   0.59    0.25    0.005    0.199           NaN      NaN  0.0               0.33   0.33   0.11   0.28    0.74    0.011    0.219   

Scenario                                              seperate_mixed_5050                                                                            cluster_aware_ignorant                       \
Metric                    Peak GPU (MB) Mat (ms)    N              SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m) Inf (ms) Peak GPU (MB) Mat (ms)    N                 SR (%) CR (%) NT (s) PL (m)   
Method                                                                                                                                                                                             
SF                                  NaN      NaN  0.0                2.99   2.99   1.64   1.05    0.58    0.012      NaN           NaN      NaN  0.0                   1.74   0.52   3.49   0.31   
ORCA                                NaN      NaN  0.0                1.91   1.48   0.41   0.32    0.14    0.013      NaN           NaN      NaN  0.0                   2.24   1.89   0.51   0.48   
CrowdNav++                          NaN      NaN  0.0                1.24   1.24   0.20   0.23    0.73    0.005      NaN           NaN      NaN  0.0                   1.57   1.57   0.11   0.21   
GenSafeNav (conservative)           NaN      NaN  NaN                 NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                    NaN    NaN    NaN    NaN   
GenSafeNav (aggressive)             NaN      NaN  NaN                 NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                    NaN    NaN    NaN    NaN   
Ours (adaptive GT)                  NaN      NaN  0.0                1.28   1.28   0.39   0.54    0.38    0.008    0.118           NaN      NaN  0.0                   0.83   0.83   0.29   0.56   

Scenario                                                                         
Metric                    ITR (%) SocD (m) Inf (ms) Peak GPU (MB) Mat (ms)    N  
Method                                                                           
SF                           0.09    0.029      NaN           NaN      NaN  0.0  
ORCA                         0.17    0.023      NaN           NaN      NaN  0.0  
CrowdNav++                   0.16    0.002      NaN           NaN      NaN  0.0  
GenSafeNav (co

In [7]:
# ============================================================
# Table C — SD ACROSS EPISODES (single seed)
#   Within one seed's test run (~250–1250 episodes), what's the std
#   over the episodes themselves?
#   - SR / CR: Bernoulli sample std = sqrt(p*(1-p))   (per-episode binary)
#   - NT     : std_nav_time             (already in summary)
#   - PL     : std_path_length          (already in summary)
#   - ITR    : std_intrusion_ratio_pct  (already in summary; already pct)
#   - SocD   : std_min_social_distance  (already in summary)
# Uses the lexicographically-smallest available seed per method.
# ============================================================

SD_KEY_FOR_MEAN = {
    'success_rate':            None,
    'collision_rate':          None,
    'avg_nav_time':            'std_nav_time',
    'avg_path_length':         'std_path_length',
    'avg_intrusion_ratio_pct': 'std_intrusion_ratio_pct',
    'avg_min_social_distance': 'std_min_social_distance',
    'avg_inference_time_ms':  'std_inference_time_ms',
    'max_inference_peak_gpu_memory_mb': None,
    'avg_matrix_calc_time_ms': 'std_matrix_calc_time_ms',
    'num_episodes': None,
}

def _real_or_nan(x):
    if x is None:
        return np.nan
    if isinstance(x, complex):
        return float(x.real) if abs(x.imag) < 1e-9 else np.nan
    return float(x)

rows_c = []
for name, _, _, seeds in METHODS:
    row = {'Method': name}
    target_seed = sorted(seeds)[0]
    for sc in SCENARIOS:
        sums = loaded[name][sc].get(target_seed, {})
        for mkey, mlabel, in_pct, dec in METRICS:
            sd_key = SD_KEY_FOR_MEAN[mkey]
            if sd_key is None:
                p = sums.get(mkey)
                if p is None:
                    sd = None
                else:
                    p = min(max(float(p), 0.0), 1.0)
                    sd = np.sqrt(p * (1.0 - p))
            else:
                sd = sums.get(sd_key)
            v = _real_or_nan(sd)
            if np.isnan(v):
                row[(sc, mlabel)] = np.nan
            else:
                v = v * 100 if in_pct else v
                row[(sc, mlabel)] = round(v, dec)
    rows_c.append(row)

df_sd_episodes = pd.DataFrame(rows_c).set_index('Method')
df_sd_episodes.columns = pd.MultiIndex.from_tuples(df_sd_episodes.columns, names=['Scenario', 'Metric'])
df_sd_episodes = df_sd_episodes.reindex(columns=pd.MultiIndex.from_product(
    [SCENARIOS, [m[1] for m in METRICS]], names=['Scenario', 'Metric']))
df_sd_episodes

Scenario                  seperate_all_ignorant                                                                            seperate_all_aware                                                 \
Metric                                   SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m) Inf (ms) Peak GPU (MB) Mat (ms)    N             SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m) Inf (ms)   
Method                                                                                                                                                                                         
SF                                        37.74  36.66  12.98  12.64    6.91    0.127      NaN           NaN      NaN  0.0              40.59   6.31  12.21   8.71    2.98    0.130      NaN   
ORCA                                      46.17  44.50  10.60   7.49    3.41    0.093      NaN           NaN      NaN  0.0              22.99   8.91   9.53   6.74    1.32    0.090      NaN   
CrowdNav++                                28.90  28.90   4.62   5.88    9.85    0.142      NaN           NaN      NaN  0.0               8.91   8.91   3.40   4.65    8.91    0.138      NaN   
GenSafeNav (conservative)                   NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                NaN    NaN    NaN    NaN     NaN      NaN      NaN   
GenSafeNav (aggressive)                     NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                NaN    NaN    NaN    NaN     NaN      NaN      NaN   
Ours (adaptive GT)                        16.50  16.50   5.46   6.31    4.10    0.114    2.076           NaN      NaN  0.0               8.91   8.91   2.33   4.03    9.83    0.134    2.657   

Scenario                                              seperate_mixed_5050                                                                            cluster_aware_ignorant                       \
Metric                    Peak GPU (MB) Mat (ms)    N              SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m) Inf (ms) Peak GPU (MB) Mat (ms)    N                 SR (%) CR (%) NT (s) PL (m)   
Method                                                                                                                                                                                             
SF                                  NaN      NaN  0.0               39.70  31.54  11.38  11.79    6.26    0.141      NaN           NaN      NaN  0.0                  33.85   8.91  11.60   7.88   
ORCA                                NaN      NaN  0.0               38.42  34.28  10.42   7.58    2.80    0.102      NaN           NaN      NaN  0.0                  29.46  24.48   9.37   6.70   
CrowdNav++                          NaN      NaN  0.0               20.51  20.51   4.04   5.08    8.78    0.136      NaN           NaN      NaN  0.0                  22.20  22.20   3.63   5.07   
GenSafeNav (conservative)           NaN      NaN  NaN                 NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                    NaN    NaN    NaN    NaN   
GenSafeNav (aggressive)             NaN      NaN  NaN                 NaN    NaN    NaN    NaN     NaN      NaN      NaN           NaN      NaN  NaN                    NaN    NaN    NaN    NaN   
Ours (adaptive GT)                  NaN      NaN  0.0               19.60  19.60   3.86   4.95    8.38    0.137    1.746           NaN      NaN  0.0                  23.75  23.75   3.78   5.48   

Scenario                                                                         
Metric                    ITR (%) SocD (m) Inf (ms) Peak GPU (MB) Mat (ms)    N  
Method                                                                           
SF                           1.19    0.152      NaN           NaN      NaN  0.0  
ORCA                         4.12    0.147      NaN           NaN      NaN  0.0  
CrowdNav++                   9.58    0.143      NaN           NaN      NaN  0.0  
GenSafeNav (co

In [8]:
# ----------------------- save outputs -----------------------
df.to_csv('giant_table_long.csv', index=False)
wide.to_csv('giant_table_wide.csv')
df_sd_scenarios.to_csv('giant_table_sd_across_scenarios.csv')
df_sd_seeds.to_csv('giant_table_sd_across_seeds.csv')
df_sd_episodes.to_csv('giant_table_sd_across_episodes_seed42.csv')

try:
    latex = wide.to_latex(na_rep='—', multicolumn=True, escape=True)
    with open('giant_table.tex', 'w') as f:
        f.write(latex)
    print('Wrote:')
    print('  giant_table_long.csv, giant_table_wide.csv, giant_table.tex')
    print('  giant_table_sd_across_scenarios.csv')
    print('  giant_table_sd_across_seeds.csv')
    print('  giant_table_sd_across_episodes_seed42.csv')
except Exception as e:
    print(f'CSVs written. LaTeX export failed: {e}')

Wrote:
  giant_table_long.csv, giant_table_wide.csv, giant_table.tex
  giant_table_sd_across_scenarios.csv
  giant_table_sd_across_seeds.csv
  giant_table_sd_across_episodes_seed42.csv


# LoRA Scale Ablation

Single source: `trained_models/LoraF_invi_visi_rank_1`. Rows are the LoRA branch's `dynamic_scale` setting:

- **scale 0.0** — LoRA branch disabled (= `always_off` = `GenSafeNav (conservative)` row above).
- **scale 0.2 / 0.4 / 0.6 / 0.8 / 1.0** — LoRA branch held at a fixed value (`--lora_behaviour fixed_scale --lora_scale X`). Test runs use `--exp_id scale_X_<seed>` so keys look like `<scenario>_fixed_scale_expscale_X_<seed>`.
- **adaptive_gt** — dynamic scale interpolated from ground-truth awareness signal (= `Ours (adaptive GT)` row above).

Each cell = `mean ± seed-SD` (SR/CR/ITR in %, NT/PL/SocD in physical units). Empty cells (`—`) mean the test for that (scale, scenario, seed) hasn't been run yet.

In [9]:
# ----------------------- LoRA ablation config -----------------------
ABL_DIR   = 'trained_models/LoraF_invi_visi_rank_1'
ABL_SEEDS = ['42', '1000', '2000', '3000', '4000']

def _scale_seeds(scale_str):
    # test_lora_ablation.sh writes --exp_id "scale_<S>_<seed>"
    return {f'scale_{scale_str}_{s}' for s in ABL_SEEDS}

# (row_label, behaviour, seed_set)
ABL_METHODS = [
    ('scale 0.0 (off)',  'always_off',  set(ABL_SEEDS)),
    ('scale 0.2',        'fixed_scale', _scale_seeds('0.2')),
    ('scale 0.4',        'fixed_scale', _scale_seeds('0.4')),
    ('scale 0.6',        'fixed_scale', _scale_seeds('0.6')),
    ('scale 0.8',        'fixed_scale', _scale_seeds('0.8')),
    ('scale 1.0',        'fixed_scale', _scale_seeds('1.0')),
    ('adaptive_gt',      'adaptive_gt', set(ABL_SEEDS)),
]

abl_loaded = {label: load_method(ABL_DIR, beh, seeds) for label, beh, seeds in ABL_METHODS}

# Coverage report
print(f"{'row':<22} {'scenario':<26} {'expect':<6} marker  found")
for label, _, seeds in ABL_METHODS:
    for sc in SCENARIOS:
        found = sorted(abl_loaded[label][sc].keys())
        marker = '✓' if seeds.issubset(found) else 'PARTIAL' if found else 'EMPTY'
        print(f"  {label:<20} {sc:<26} {len(seeds):<6} {marker:<7} {found}")

row                    scenario                   expect marker  found
  scale 0.0 (off)      seperate_all_ignorant      5      EMPTY   []
  scale 0.0 (off)      seperate_all_aware         5      EMPTY   []
  scale 0.0 (off)      seperate_mixed_5050        5      EMPTY   []
  scale 0.0 (off)      cluster_aware_ignorant     5      EMPTY   []
  scale 0.2            seperate_all_ignorant      5      EMPTY   []
  scale 0.2            seperate_all_aware         5      EMPTY   []
  scale 0.2            seperate_mixed_5050        5      EMPTY   []
  scale 0.2            cluster_aware_ignorant     5      EMPTY   []
  scale 0.4            seperate_all_ignorant      5      EMPTY   []
  scale 0.4            seperate_all_aware         5      EMPTY   []
  scale 0.4            seperate_mixed_5050        5      EMPTY   []
  scale 0.4            cluster_aware_ignorant     5      EMPTY   []
  scale 0.6            seperate_all_ignorant      5      EMPTY   []
  scale 0.6            seperate_all_aware    

In [10]:
# ----------------------- ablation: long view (separate mean / seed-SD columns) -----------------------
# Ablation has one extra column vs. the main table: the average LoRA scale
# (`dynamic_scale`) actually applied per step, averaged over the test episodes.
# For fixed-scale rows this equals the fixed value; for adaptive_gt it reflects
# the time-averaged interpolated scale.
ABL_METRICS = METRICS + [('avg_lora_scale', 'LoRA scale', False, 3)]

abl_rows = []
for label, _, _ in ABL_METHODS:
    for sc in SCENARIOS:
        seed_dicts = abl_loaded[label][sc]
        row = {'Row': label, 'Scenario': sc}
        for mkey, mlabel, in_pct, dec in ABL_METRICS:
            vals = [s.get(mkey) for s in seed_dicts.values() if s.get(mkey) is not None]
            add_metric_stats(row, mkey, mlabel, in_pct, dec, vals)
        abl_rows.append(row)

df_ablation = pd.DataFrame(abl_rows)
df_ablation


,Row,Scenario,SR (%) mean,SR (%) std,CR (%) mean,CR (%) std,NT (s) mean,NT (s) std,PL (m) mean,PL (m) std,ITR (%) mean,ITR (%) std,SocD (m) mean,SocD (m) std,Inf (ms) mean,Inf (ms) std,Peak GPU (MB) mean,Peak GPU (MB) std,Mat (ms) mean,Mat (ms) std,N total,LoRA scale mean,LoRA scale std
0,scale 0.0 (off),seperate_all_ignorant,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
1,scale 0.0 (off),seperate_all_aware,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
2,scale 0.0 (off),seperate_mixed_5050,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
3,scale 0.0 (off),cluster_aware_ignorant,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
4,scale 0.2,seperate_all_ignorant,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
5,scale 0.2,seperate_all_aware,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
6,scale 0.2,seperate_mixed_5050,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
7,scale 0.2,cluster_aware_ignorant,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
8,scale 0.4,seperate_all_ignorant,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—
9,scale 0.4,seperate_all_aware,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—,—


In [11]:
# ----------------------- ablation: scenario-first pivot -----------------------
value_cols = [c for c in df_ablation.columns if c not in ('Row', 'Scenario')]
abl_long = df_ablation.melt(id_vars=['Row', 'Scenario'], value_vars=value_cols, var_name='MetricStat', value_name='Value')
abl_long[['Metric', 'Stat']] = abl_long['MetricStat'].str.rsplit(' ', n=1, expand=True)
abl_long['Scenario group'] = abl_long['Scenario'].map(SCENARIO_LABELS).fillna(abl_long['Scenario'])

abl_wide = abl_long.pivot_table(index=['Scenario group', 'Row'], columns=['Metric', 'Stat'], values='Value', aggfunc='first')
abl_wide = abl_wide.reindex(pd.MultiIndex.from_tuples(
    [(SCENARIO_LABELS[sc], method) for sc in SCENARIO_DISPLAY_ORDER for method, _, _ in ABL_METHODS],
    names=['Scenario', 'Method'],
))
abl_wide = abl_wide.reindex(columns=pd.MultiIndex.from_tuples(
    metric_stat_columns(ABL_METRICS),
    names=['Metric', 'Stat'],
))

df_ablation.to_csv('giant_table_ablation_long.csv', index=False)
abl_wide.to_csv('giant_table_ablation_wide.csv')
print('Wrote: giant_table_ablation_long.csv, giant_table_ablation_wide.csv')
abl_wide


Wrote: giant_table_ablation_long.csv, giant_table_ablation_wide.csv


Metric                           SR (%)       CR (%)       NT (s)       PL (m)       ITR (%)       SocD (m)        Inf (ms)        Peak GPU (MB)     Mat (ms)         N LoRA scale       
Stat                               mean   std   mean   std   mean   std   mean   std    mean   std     mean    std     mean    std          mean std     mean std total       mean    std
Scenario         Method                                                                                                                                                                  
Non-aware        scale 0.0 (off)      —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.2            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.4            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.6            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.8            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 1.0            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 adaptive_gt      97.04  0.36   2.96  0.36  16.51  0.44  23.44  0.59    2.39  0.25    0.448  0.005    5.856  0.199             —   —        —   —  1250      0.017  0.006
Aware            scale 0.0 (off)      —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.2            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.4            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.6            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.8            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 1.0            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 adaptive_gt      99.28  0.33   0.72  0.33  10.94  0.11  18.16  0.28    7.33  0.74    0.417  0.011    3.767  0.219             —   —        —   —  1250      1.000  0.000
Mixed            scale 0.0 (off)      —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.2            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.4            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.6            —     —      —     —      —     —      —     —       —     —        —      —        —      —             —   —        —   —     —          —      —
                 scale 0.8            —     —      —     —      —     —      —     —       —  

# Full-Finetune Scale Ablation

Dense interpolation sweep from the conservative backbone to the full-finetuned endpoint. Source policy is `trained_models/LoraF_invi_visi_rank_1`; dense endpoint is `trained_models/Fullfinetune_invi_visi_new/checkpoints/03400.pt`.

Run with:

```bash
./test_fullfinetune_ablation.sh
```

Rows are fixed dense interpolation values (`--lora_behaviour fixed_fullfinetune_scale --lora_scale X`) plus the dynamic `adaptive_fullfinetune_gt` baseline. Empty cells (`—`) mean that result has not been run yet.


In [12]:
# ----------------------- full-finetune ablation config -----------------------
FULLFT_DIR = 'trained_models/LoraF_invi_visi_rank_1'
FULLFT_SEEDS = ['42', '1000', '2000', '3000', '4000']

def _fullft_scale_seeds(scale_str):
    # test_fullfinetune_ablation.sh writes --exp_id "fullft_scale_<S>_<seed>"
    return {f'fullft_scale_{scale_str}_{s}' for s in FULLFT_SEEDS}

FULLFT_METHODS = [
    ('fullft scale 0.0', 'fixed_fullfinetune_scale', _fullft_scale_seeds('0.0')),
    ('fullft scale 0.2', 'fixed_fullfinetune_scale', _fullft_scale_seeds('0.2')),
    ('fullft scale 0.4', 'fixed_fullfinetune_scale', _fullft_scale_seeds('0.4')),
    ('fullft scale 0.6', 'fixed_fullfinetune_scale', _fullft_scale_seeds('0.6')),
    ('fullft scale 0.8', 'fixed_fullfinetune_scale', _fullft_scale_seeds('0.8')),
    ('fullft scale 1.0', 'fixed_fullfinetune_scale', _fullft_scale_seeds('1.0')),
    ('adaptive_fullfinetune_gt', 'adaptive_fullfinetune_gt', set(FULLFT_SEEDS)),
]

fullft_loaded = {label: load_method(FULLFT_DIR, beh, seeds) for label, beh, seeds in FULLFT_METHODS}

print(f"{'row':<26} {'scenario':<26} {'expect':<6} marker  found")
for label, _, seeds in FULLFT_METHODS:
    for sc in SCENARIOS:
        found = sorted(fullft_loaded[label][sc].keys())
        marker = '✓' if seeds.issubset(found) else 'PARTIAL' if found else 'EMPTY'
        print(f"  {label:<24} {sc:<26} {len(seeds):<6} {marker:<7} {found}")


row                        scenario                   expect marker  found
  fullft scale 0.0         seperate_all_ignorant      5      ✓       ['fullft_scale_0.0_1000', 'fullft_scale_0.0_2000', 'fullft_scale_0.0_3000', 'fullft_scale_0.0_4000', 'fullft_scale_0.0_42']
  fullft scale 0.0         seperate_all_aware         5      ✓       ['fullft_scale_0.0_1000', 'fullft_scale_0.0_2000', 'fullft_scale_0.0_3000', 'fullft_scale_0.0_4000', 'fullft_scale_0.0_42']
  fullft scale 0.0         seperate_mixed_5050        5      ✓       ['fullft_scale_0.0_1000', 'fullft_scale_0.0_2000', 'fullft_scale_0.0_3000', 'fullft_scale_0.0_4000', 'fullft_scale_0.0_42']
  fullft scale 0.0         cluster_aware_ignorant     5      ✓       ['fullft_scale_0.0_1000', 'fullft_scale_0.0_2000', 'fullft_scale_0.0_3000', 'fullft_scale_0.0_4000', 'fullft_scale_0.0_42']
  fullft scale 0.2         seperate_all_ignorant      5      ✓       ['fullft_scale_0.2_1000', 'fullft_scale_0.2_2000', 'fullft_scale_0.2_3000', 'fullft_

In [13]:
# ----------------------- full-finetune ablation: long view (separate mean / seed-SD columns) -----------------------
FULLFT_METRICS = METRICS + [('avg_lora_scale', 'Dense scale', False, 3)]

fullft_rows = []
for label, _, _ in FULLFT_METHODS:
    for sc in SCENARIOS:
        seed_dicts = fullft_loaded[label][sc]
        row = {'Row': label, 'Scenario': sc}
        for mkey, mlabel, in_pct, dec in FULLFT_METRICS:
            vals = [s.get(mkey) for s in seed_dicts.values() if s.get(mkey) is not None]
            add_metric_stats(row, mkey, mlabel, in_pct, dec, vals)
        fullft_rows.append(row)

df_fullfinetune_ablation = pd.DataFrame(fullft_rows)
df_fullfinetune_ablation


,Row,Scenario,SR (%) mean,SR (%) std,CR (%) mean,CR (%) std,NT (s) mean,NT (s) std,PL (m) mean,PL (m) std,ITR (%) mean,ITR (%) std,SocD (m) mean,SocD (m) std,Inf (ms) mean,Inf (ms) std,Peak GPU (MB) mean,Peak GPU (MB) std,Mat (ms) mean,Mat (ms) std,N total,Dense scale mean,Dense scale std
0,fullft scale 0.0,seperate_all_ignorant,97.20,0.40,2.80,0.40,16.59,0.47,23.53,0.62,2.30,0.29,0.451,0.004,7.531,3.137,—,—,—,—,1250,0.000,0.000
1,fullft scale 0.0,seperate_all_aware,100.00,0.00,0.00,0.00,16.24,0.47,23.49,0.64,2.61,0.32,0.439,0.011,3.345,1.404,—,—,—,—,1250,0.000,0.000
2,fullft scale 0.0,seperate_mixed_5050,98.56,0.73,1.36,0.73,16.61,0.58,23.79,0.74,2.62,0.23,0.439,0.010,7.280,3.654,—,—,—,—,1250,0.000,0.000
3,fullft scale 0.0,cluster_aware_ignorant,97.12,0.77,2.88,0.77,15.47,0.68,22.40,0.89,3.89,0.40,0.431,0.002,3.459,1.430,—,—,—,—,1250,0.000,0.000
4,fullft scale 0.2,seperate_all_ignorant,89.60,3.59,10.40,3.59,14.17,0.28,20.77,0.54,5.38,0.42,0.438,0.002,7.458,2.242,—,—,—,—,1250,0.200,0.000
5,fullft scale 0.2,seperate_all_aware,99.76,0.54,0.24,0.54,13.06,0.22,20.33,0.35,5.02,0.67,0.436,0.007,3.149,1.190,—,—,—,—,1250,0.200,0.000
6,fullft scale 0.2,seperate_mixed_5050,96.56,1.22,3.44,1.22,13.93,0.36,20.99,0.51,5.35,0.46,0.436,0.014,7.414,2.587,—,—,—,—,1250,0.200,0.000
7,fullft scale 0.2,cluster_aware_ignorant,93.44,1.46,6.56,1.46,12.97,0.16,19.90,0.36,7.07,0.57,0.426,0.008,3.175,1.270,—,—,—,—,1250,0.200,0.000
8,fullft scale 0.4,seperate_all_ignorant,73.76,1.76,26.24,1.76,12.52,0.19,18.42,0.29,10.45,0.67,0.430,0.008,3.343,1.201,—,—,—,—,1250,0.400,0.000
9,fullft scale 0.4,seperate_all_aware,99.84,0.36,0.16,0.36,11.23,0.05,18.50,0.23,7.24,0.64,0.428,0.009,7.414,3.865,—,—,—,—,1250,0.400,0.000


In [14]:
# ----------------------- full-finetune ablation: scenario-first pivot -----------------------
value_cols = [c for c in df_fullfinetune_ablation.columns if c not in ('Row', 'Scenario')]
fullft_long = df_fullfinetune_ablation.melt(id_vars=['Row', 'Scenario'], value_vars=value_cols, var_name='MetricStat', value_name='Value')
fullft_long[['Metric', 'Stat']] = fullft_long['MetricStat'].str.rsplit(' ', n=1, expand=True)
fullft_long['Scenario group'] = fullft_long['Scenario'].map(SCENARIO_LABELS).fillna(fullft_long['Scenario'])

fullft_wide = fullft_long.pivot_table(index=['Scenario group', 'Row'], columns=['Metric', 'Stat'], values='Value', aggfunc='first')
fullft_wide = fullft_wide.reindex(pd.MultiIndex.from_tuples(
    [(SCENARIO_LABELS[sc], method) for sc in SCENARIO_DISPLAY_ORDER for method, _, _ in FULLFT_METHODS],
    names=['Scenario', 'Method'],
))
fullft_wide = fullft_wide.reindex(columns=pd.MultiIndex.from_tuples(
    metric_stat_columns(FULLFT_METRICS),
    names=['Metric', 'Stat'],
))

df_fullfinetune_ablation.to_csv('giant_table_fullfinetune_ablation_long.csv', index=False)
fullft_wide.to_csv('giant_table_fullfinetune_ablation_wide.csv')
print('Wrote: giant_table_fullfinetune_ablation_long.csv, giant_table_fullfinetune_ablation_wide.csv')
fullft_wide


Wrote: giant_table_fullfinetune_ablation_long.csv, giant_table_fullfinetune_ablation_wide.csv


Metric                                     SR (%)       CR (%)       NT (s)       PL (m)       ITR (%)       SocD (m)        Inf (ms)        Peak GPU (MB)     Mat (ms)         N Dense scale       
Stat                                         mean   std   mean   std   mean   std   mean   std    mean   std     mean    std     mean    std          mean std     mean std total        mean    std
Scenario         Method                                                                                                                                                                             
Non-aware        fullft scale 0.0           97.20  0.40   2.80  0.40  16.59  0.47  23.53  0.62    2.30  0.29    0.451  0.004    7.531  3.137             —   —        —   —  1250       0.000  0.000
                 fullft scale 0.2           89.60  3.59  10.40  3.59  14.17  0.28  20.77  0.54    5.38  0.42    0.438  0.002    7.458  2.242             —   —        —   —  1250       0.200  0.000
                 fullft scale 0.4           73.76  1.76  26.24  1.76  12.52  0.19  18.42  0.29   10.45  0.67    0.430  0.008    3.343  1.201             —   —        —   —  1250       0.400  0.000
                 fullft scale 0.6           61.20  4.18  38.80  4.18  11.31  0.16  16.76  0.20   13.63  1.07    0.420  0.005    2.962  0.951             —   —        —   —  1250       0.600  0.000
                 fullft scale 0.8           50.80  2.53  49.20  2.53  10.68  0.16  15.70  0.10   14.80  0.87    0.413  0.003    7.516  3.401             —   —        —   —  1250       0.800  0.000
                 fullft scale 1.0           46.24  1.49  53.76  1.49  10.62  0.17  15.41  0.17   15.29  0.90    0.409  0.004    7.356  2.261             —   —        —   —  1250       1.000  0.000
                 adaptive_fullfinetune_gt   97.04  0.61   2.96  0.61  16.51  0.46  23.43  0.65    2.39  0.25    0.449  0.003    3.753  0.649             —   —        —   —  1250       0.018  0.006
Aware            fullft scale 0.0          100.00  0.00   0.00  0.00  16.24  0.47  23.49  0.64    2.61  0.32    0.439  0.011    3.345  1.404             —   —        —   —  1250       0.000  0.000
                 fullft scale 0.2           99.76  0.54   0.24  0.54  13.06  0.22  20.33  0.35    5.02  0.67    0.436  0.007    3.149  1.190             —   —        —   —  1250       0.200  0.000
                 fullft scale 0.4           99.84  0.36   0.16  0.36  11.23  0.05  18.50  0.23    7.24  0.64    0.428  0.009    7.414  3.865             —   —        —   —  1250       0.400  0.000
                 fullft scale 0.6           99.28  0.66   0.72  0.66  10.53  0.08  17.78  0.24    8.86  0.71    0.421  0.009    7.381  2.438             —   —        —   —  1250       0.600  0.000
                 fullft scale 0.8           98.80  0.80   1.20  0.80  10.25  0.09  17.45  0.26    9.57  0.81    0.416  0.009    3.367  1.379             —   —        —   —  1250       0.800  0.000
                 fullft scale 1.0           98.96  0.54   1.04  0.54  10.27  0.07  17.48  0.17    9.65  0.82    0.417  0.011    3.201  1.169             —   —        —   —  1250       1.000  0.000
                 adaptive_fullfinetune_gt   99.28  0.33   0.72  0.33  10.69  0.07  17.93  0.21    8.14  0.71    0.429  0.006    4.629  0.471             —   —        —   —  1250       1.000  0.000
Mixed            fullft scale 0.0           98.56  0.73   1.36  0.73  16.61  0.58  23.79  0.74    2.62  0.23    0.439  0.010    7.280  3.654             —   —        —   —  1250       0.000  0.000
                 fullft scale 0.2           96.56  1.22   3.44  1.22  13.93  0.36  20.99  0.51    5.35  0.46    0.436  0.014    7.414  2.587             —   —        —   —  1250       0.200  0.000
                 fullft scale 0.4           89.04  0.36  10.96  0.36  12.09  0.12  18.90  0.27    9.03  0.64    0.429  0.009    3.399  1.451             —   —        —   —  1250       0.400  0.000
                 fullft scale 0.6           80.88  1.66  19.

# Rebuttal

Action-space, dense-full-finetune interpolation, and conservative up-cost baselines for the rebuttal. Run this sweep first so the relevant `trained_models/*/test/all_evaluations.json` files contain entries across all scenarios and seeds.

```bash
BLEND_GPUS=0,1,2,3 \
ADAPTIVE_MODEL="trained_models/LoraF_invi_visi_rank_4" \
CHECKPOINT="03400.pt" \
EXP_NOTE="timetest" \
SEEDS="42 1000 2000 3000 4000" \
ADAPTIVE_BEHAVIOURS="adaptive_gt adaptive_action_gt adaptive_fullfinetune_gt Gensafenav_cons_upcost" \
TEST_SIZE=250 \
HUMAN_NUM=20 \
AWARENESS_EVAL=off \
MAX_PARALLEL=4 \
./test_adaptive_lora_poc.sh

# EXP_NOTE=timetest writes keys like <scenario>_<behaviour>_exptimetest_42.
```

In [ ]:
# ----------------------- Rebuttal: adaptive baselines -----------------------
import os, json
import numpy as np
import pandas as pd

REBUTTAL_METHODS = [
    ('Adaptive GT', 'trained_models/LoraF_invi_visi_rank_1', 'adaptive_gt'),
    ('Action-space adaptive GT', 'trained_models/LoraF_invi_visi_rank_1', 'adaptive_action_gt'),
    ('Adaptive full-finetune GT', 'trained_models/LoraF_invi_visi_rank_1', 'adaptive_fullfinetune_gt'),
    ('Gensafenav_cons_upcost', 'trained_models/Conservative_Backbone_CostLimit_1.2', 'Gensafenav_cons_upcost'),
]
REBUTTAL_SEEDS = ['42', '1000', '2000', '3000', '4000']
# Exp id pattern used in all_evaluations.json keys. Default '{seed}' matches
# keys like '<scenario>_<behaviour>_exp42'. For tagged per-seed runs, set
# e.g. REBUTTAL_EXP_ID_TEMPLATE = 'rank4_{seed}'.
REBUTTAL_EXP_ID_TEMPLATE = '{seed}'
# If non-empty, this overrides REBUTTAL_EXP_ID_TEMPLATE and reads keys like
# '<scenario>_<behaviour>_exp<note>_<seed>'. Match this to EXP_NOTE.
REBUTTAL_EXP_NOTE = 'timetest'
REBUTTAL_SCENARIOS = SCENARIOS
REBUTTAL_METRICS = [
    ('success_rate', 'SR (%)', True, 2),
    ('collision_rate', 'CR (%)', True, 2),
    ('avg_nav_time', 'NT (s)', False, 2),
    ('avg_path_length', 'PL (m)', False, 2),
    ('avg_intrusion_ratio_pct', 'ITR (%)', False, 2),
    ('avg_min_social_distance', 'SocD (m)', False, 3),
    ('avg_lora_scale', 'Avg kappa', False, 3),
    ('avg_inference_time_ms', 'Inf (ms)', False, 3),
    ('p95_inference_time_ms', 'P95 Inf (ms)', False, 3),
    ('max_inference_peak_gpu_memory_mb', 'Peak GPU (MB)', False, 1),
    ('avg_matrix_calc_time_ms', 'Mat (ms)', False, 3),
    ('p95_matrix_calc_time_ms', 'P95 Mat (ms)', False, 3),
    ('num_episodes', 'N', False, 0),
]

_aggregate_cache = {}
def _load_rebuttal_aggregate(model_dir):
    if model_dir not in _aggregate_cache:
        agg_path = os.path.join(model_dir, 'test', 'all_evaluations.json')
        if not os.path.exists(agg_path):
            _aggregate_cache[model_dir] = ({}, agg_path)
        else:
            with open(agg_path) as f:
                _aggregate_cache[model_dir] = (json.load(f), agg_path)
    return _aggregate_cache[model_dir]

coverage_rows = []
table_rows = []
for method_name, model_dir, behaviour in REBUTTAL_METHODS:
    aggregate, agg_path = _load_rebuttal_aggregate(model_dir)
    for scenario in REBUTTAL_SCENARIOS:
        summaries = {}
        for seed in REBUTTAL_SEEDS:
            exp_id = f'{REBUTTAL_EXP_NOTE}_{seed}' if REBUTTAL_EXP_NOTE else REBUTTAL_EXP_ID_TEMPLATE.format(seed=seed)
            key = f'{scenario}_{behaviour}_exp{exp_id}'
            ent = aggregate.get(key)
            if ent is not None:
                summaries[seed] = ent.get('summary', {})

        found = sorted(summaries.keys(), key=lambda x: REBUTTAL_SEEDS.index(x))
        coverage_rows.append({
            'Method': method_name,
            'Behaviour': behaviour,
            'Model dir': model_dir,
            'Aggregate': agg_path,
            'Scenario': SCENARIO_LABELS.get(scenario, scenario),
            'Exp note': REBUTTAL_EXP_NOTE,
            'Exp id template': REBUTTAL_EXP_ID_TEMPLATE,
            'Expected seeds': len(REBUTTAL_SEEDS),
            'Found seeds': len(found),
            'Complete': len(found) == len(REBUTTAL_SEEDS),
            'Seeds': found,
        })

        row = {'Method': method_name, 'Behaviour': behaviour, 'Scenario': scenario}
        for metric_key, label, in_percent, decimals in REBUTTAL_METRICS:
            vals = [s.get(metric_key) for s in summaries.values() if s.get(metric_key) is not None]
            add_metric_stats(row, metric_key, label, in_percent, decimals, vals)
        table_rows.append(row)

rebuttal_coverage = pd.DataFrame(coverage_rows)
display(rebuttal_coverage)

df_rebuttal_adaptive_baselines = pd.DataFrame(table_rows)
value_cols = [c for c in df_rebuttal_adaptive_baselines.columns if c not in ('Method', 'Behaviour', 'Scenario')]
rebuttal_long = df_rebuttal_adaptive_baselines.melt(
    id_vars=['Method', 'Behaviour', 'Scenario'],
    value_vars=value_cols,
    var_name='MetricStat',
    value_name='Value',
)

rebuttal_long[['Metric', 'Stat']] = rebuttal_long['MetricStat'].str.rsplit(' ', n=1, expand=True)
rebuttal_long['Scenario group'] = rebuttal_long['Scenario'].map(SCENARIO_LABELS).fillna(rebuttal_long['Scenario'])
rebuttal_wide = rebuttal_long.pivot_table(index=['Scenario group', 'Method'], columns=['Metric', 'Stat'], values='Value', aggfunc='first')
rebuttal_wide = rebuttal_wide.reindex(pd.MultiIndex.from_tuples(
    [(SCENARIO_LABELS[sc], method_name)
     for sc in REBUTTAL_SCENARIOS
     for method_name, _, _ in REBUTTAL_METHODS],
    names=['Scenario', 'Method'],
))

rebuttal_wide = rebuttal_wide.reindex(columns=pd.MultiIndex.from_tuples(
    metric_stat_columns(REBUTTAL_METRICS),
    names=['Metric', 'Stat'],
))

df_rebuttal_adaptive_baselines.to_csv('giant_table_rebuttal_adaptive_baselines.csv', index=False)
rebuttal_wide.to_csv('giant_table_rebuttal_adaptive_baselines_wide.csv')
print('Wrote: giant_table_rebuttal_adaptive_baselines.csv, giant_table_rebuttal_adaptive_baselines_wide.csv')
rebuttal_wide


,Method,Behaviour,Model dir,Aggregate,Scenario,Exp note,Exp id template,Expected seeds,Found seeds,Complete,Seeds
0,Adaptive GT,adaptive_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Non-aware,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
1,Adaptive GT,adaptive_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Aware,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
2,Adaptive GT,adaptive_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Mixed,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
3,Adaptive GT,adaptive_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Spatial Clusters,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
4,Action-space adaptive GT,adaptive_action_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Non-aware,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
5,Action-space adaptive GT,adaptive_action_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Aware,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
6,Action-space adaptive GT,adaptive_action_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Mixed,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
7,Action-space adaptive GT,adaptive_action_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Spatial Clusters,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
8,Adaptive full-finetune GT,adaptive_fullfinetune_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Non-aware,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"
9,Adaptive full-finetune GT,adaptive_fullfinetune_gt,trained_models/LoraF_invi_visi_rank_1,trained_models/LoraF_invi_visi_rank_1/test/all...,Aware,,{seed},5,5,True,"[42, 1000, 2000, 3000, 4000]"


Wrote: giant_table_rebuttal_adaptive_baselines.csv, giant_table_rebuttal_adaptive_baselines_wide.csv


Metric                                     SR (%)       CR (%)       NT (s)       PL (m)       ITR (%)       SocD (m)        Avg kappa        Inf (ms)        P95 Inf (ms)        Peak GPU (MB)      \
Stat                                         mean   std   mean   std   mean   std   mean   std    mean   std     mean    std      mean    std     mean    std         mean    std          mean std   
Scenario         Method                                                                                                                                                                               
Non-aware        Adaptive GT                97.04  0.36   2.96  0.36  16.51  0.44  23.44  0.59    2.39  0.25    0.448  0.005     0.017  0.006    5.856  0.199        8.645  0.257             —   —   
                 Action-space adaptive GT   96.96  0.54   2.96  0.46  16.49  0.49  23.44  0.62    2.36  0.24    0.449  0.003     0.017  0.006        —      —            —      —             —   —   
                 Adaptive full-finetune GT  97.04  0.61   2.96  0.61  16.51  0.46  23.43  0.65    2.39  0.25    0.449  0.003     0.018  0.006    3.753  0.649        6.286  0.718             —   —   
                 Gensafenav_cons_upcost     92.64  2.17   7.28  2.18  13.76  0.28  20.45  0.24    7.83  0.57    0.405  0.008     1.000  0.000    3.240  0.418        5.696  0.654             —   —   
Aware            Adaptive GT                99.28  0.33   0.72  0.33  10.94  0.11  18.16  0.28    7.33  0.74    0.417  0.011     1.000  0.000    3.767  0.219        7.573  0.566             —   —   
                 Action-space adaptive GT   99.28  0.33   0.72  0.33  10.94  0.11  18.16  0.28    7.33  0.74    0.417  0.011     1.000  0.000        —      —            —      —             —   —   
                 Adaptive full-finetune GT  99.28  0.33   0.72  0.33  10.69  0.07  17.93  0.21    8.14  0.71    0.429  0.006     1.000  0.000    4.629  0.471        8.005  0.813             —   —   
                 Gensafenav_cons_upcost     99.92  0.18   0.00  0.00  12.16  0.27  19.37  0.30    6.49  0.55    0.411  0.006     1.000  0.000    4.066  0.348        7.455  0.633             —   —   
Mixed            Adaptive GT                95.44  1.28   4.56  1.28  13.68  0.39  20.66  0.54    5.87  0.38    0.430  0.008     0.502  0.005    1.908  0.118        3.469  0.165             —   —   
                 Action-space adaptive GT   94.80  1.55   5.20  1.55  13.54  0.34  20.47  0.48    5.70  0.58    0.428  0.006     0.503  0.007        —      —            —      —             —   —   
                 Adaptive full-finetune GT  92.96  1.28   7.04  1.28  12.94  0.29  19.83  0.38    7.15  0.44    0.430  0.007     0.499  0.005    3.454  0.491        5.453  0.579             —   —   
                 Gensafenav_cons_upcost     97.52  0.52   2.48  0.52  13.18  0.16  20.20  0.35    7.13  0.45    0.406  0.008     1.000  0.000    2.982  0.282        5.065  0.350             —   —   
Spatial Clusters Adaptive GT                94.96  0.83   5.04  0.83  12.84  0.29  19.84  0.56    6.47  0.68    0.423  0.010     0.556  0.012    1.907  0.148        3.379  0.459             —   —   
                 Action-space adaptive GT   95.92  1.11   4.08  1.11  12.97  0.28  19.95  0.53    6.49  0.79    0.422  0.011     0.554  0.013        —      —            —      —             —   —   
                 Adaptive full-finetune GT  94.48  1.43   5.52  1.43  12.53  0.29  19.47  0.50    6.80  0.89    0.428  0.003     0.557  0.016    3.522  0.522        5.555  0.640             —   —   
                 Gensafenav_cons_upcost     97.92  0.52   2.08  0.52  13.22  0.19  20.20  0.27    6.27  0.54    0.401  0.006     1.000  0.000    3.003  0.247        5.079  0.491             —   —   

Metric                                     Mat (ms)     P95 Mat (ms)         N  
Stat                                           mean std         mean std total  
Scenario         Method                                 

## Notes

- **Aggressive row caveat:** "GenSafeNav (aggressive)" is currently sourced from `trained_models/FullFineTune_invi_visi` (`always_off`) rather than a true `always_on` LoRA run. FullFineTune is a different method (no LoRA gating — the entire policy was retrained on invi→visi), so treat this row as the *upper-bound aggressive baseline* rather than literally "GenSafeNav with the LoRA branch always on". To get a true LoRA `always_on` row later, run:
  ```bash
  for s in 42 1000 2000 3000 4000; do
    for sc in seperate_all_aware seperate_all_ignorant seperate_mixed_5050 cluster_aware_ignorant; do
      python test.py --model_dir trained_models/LoraF_invi_visi_rank_1 --test_model 03400.pt \
        --lora_behaviour always_on --adaptive_lora_scenario $sc \
        --seed $s --exp_id ${sc}_always_on_exp${s}
    done
  done
  ```
  then point the aggressive row in cell 1 back to `LoraF_invi_visi_rank_1` with behaviour `always_on`.

- **Baseline SD = 0:** SF / ORCA / CrowdNav++ rows show `± 0.0` for every metric because their `all_evaluations.json` entries are bit-identical across exp_ids 42/1000/2000/3000/4000 — those test runs were never reseeded (same simulator state, just relabelled). Re-test them with `--seed $SEED` actually wired in (the way `run_all_results.sh` does for LoraF) and the ± numbers will populate.